### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #11

# Fine-Tuning Qwen3 for Thai Text Reasoning

Today, you will learn how to take a pre-trained Large Language Model (LLM) and specialize it for **Thai text reasoning** using the [Thai Reasoning Dataset](https://huggingface.co/datasets/iapp/Thai-R1-Distill-SFT).

We will be using [**Unsloth**](https://docs.unsloth.ai/get-started/all-our-models) to speeds up finetuning and reduces memory usage, making it possible to train in Google Colab.

If GPUs are not available in your Colab, you might want to try these two alternatives:
1. [Kaggle](https://www.kaggle.com)
2. [lightning.ai](https://lightning.ai)

### Installation

In [1]:
%%capture
import os
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Fine-tuning Qwen3 with Unsloth

#### Transformer in Qwen3 vs GPT-2
<img src="https://substackcdn.com/image/fetch/$s_!GGk9!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fa8fa602a-aa4d-4526-9252-c2f09dd5de92_1871x1920.png" alt="transformers" width="600"/>

[Source: [Sebastian Raschka](https://magazine.sebastianraschka.com/p/qwen3-from-scratch)]

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 02-21 08:55:23 [__init__.py:244] Automatically detected platform cuda.
ERROR 02-21 08:55:25 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.32G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

#### Low-Rank Adaptation (LoRA)
<img src="https://substackcdn.com/image/fetch/$s_!LXL5!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F7ab94b67-4efa-45e2-9a77-3121d6c88c45_1284x364.png" alt="LoRA" width="600"/>

[Source: [Dasha Herrmannova](https://oneminutenlp.com/p/low-rank-adaptation)]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep
Use the [Thai Reasoning Dataset](https://huggingface.co/datasets/iapp/Thai-R1-Distill-SFT) provided by [iApp Technology](https://iapp.co.th/).

In [4]:
from datasets import load_dataset
reasoning_dataset = load_dataset("iapp/Thai-R1-Distill-SFT", split = "train")

README.md:   0%|          | 0.00/231 [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Let's see the structure of both datasets:

In [5]:
reasoning_dataset

Dataset({
    features: ['reannotated_assistant_content', 'problem', 'solution', 'id', 'source', 'verified', 'quality_metrics'],
    num_rows: 10000
})

We now convert the reasoning dataset into conversational format:

In [6]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

In [7]:
def format_dataset(x):
    expected_answer = x["solution"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["reannotated_assistant_content"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

prepared_dataset = reasoning_dataset.to_pandas()
prepared_dataset["Messages"] = prepared_dataset.apply(format_dataset, axis = 1)

In [8]:
prepared_dataset

,reannotated_assistant_content,problem,solution,id,source,verified,quality_metrics,Messages
0,<think>ก่อนอื่น ฉันต้องคำนวณจำนวนเด็กทั้งหมดใน...,มีเด็กชาย 27 คนและเด็กหญิง 35 คนอยู่ในสนามเด็ก...,\nมีเด็ก 62 คนอยู่ในสนามเด็กเล่นในช่วงพัก (เด็...,id_0,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
1,<think>ก่อนอื่น ฉันต้องคำนวณต้นทุนต่อส้มหนึ่งโ...,จอห์นซื้อส้มสามโหลในราคา 28.80 ดอลลาร์ หากคิดใ...,ปัญหาระบุว่าจอห์นซื้อส้มสามโหลในราคา $\$$28.80...,id_1,synthetic_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
2,<think>ขั้นแรก ให้พิจารณาว่า Bianca รีไซเคิลถุ...,บิอังกาได้รับ 5 คะแนนสำหรับกระป๋องแต่ละถุงที่เ...,Bianca รีไซเคิลกระป๋องได้ 17 - 8 = 9 ถุง สำหรั...,id_2,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
3,<think>ขั้นแรก ให้ระบุต้นทุนของฝาขวดหนึ่งฝา ซึ...,ฝาขวดแต่ละฝาราคา 2 เหรียญ ฝาขวด 6 ฝาราคาเท่าไร?,\nหากฝาขวดแต่ละฝามีราคา 2 เหรียญ ดังนั้นฝาขวด ...,id_3,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
4,<think>ก่อนอื่น ฉันต้องกำหนดว่าแจ็คได้รับอีเมล...,แจ็คได้รับอีเมล 6 ฉบับในตอนเช้าและอีเมลอีกจำนว...,\nหากแจ็คได้รับอีเมล 6 ฉบับในตอนเช้า และเขาได้...,id_4,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
...,...,...,...,...,...,...,...,...
9995,<think>ขั้นแรก ฉันต้องกำหนดจำนวนคนงานทั้งหมดใน...,เงินเดือนเฉลี่ยของคนงานทั้งหมดในโรงงานแห่งหนึ่...,ให้เราแทนเงินเดือนโดยเฉลี่ยของคนงานทั้งหมดในโร...,id_9995,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
9996,<think>ก่อนอื่น ฉันต้องเข้าใจปัญหาก่อน คนๆ หนึ...,โดยการเดินทางด้วยความเร็ว 40 กิโลเมตรต่อชั่วโม...,ให้แทนระยะทางทั้งหมดเป็น D และเวลาทั้งหมดเป็น ...,id_9996,orca_math,None,None,"[{'role': 'system', 'content': 'You are given ..."
9997,<think>โอเค ฉันมีปัญหานี้เกี่ยวกับลำดับ {a_n} ...,"ในลำดับ $\{a_{n}\}$, ${a_4}=1$, ${a_6}=\frac{1...",เนื่องจากลำดับ $\{a_{n}\}$ มีคุณสมบัติที่ $\le...,id_9997,cn_k12,None,None,"[{'role': 'system', 'content': 'You are given ..."
9998,<think>ก่อนอื่น เรามากำหนดความกว้างของสนามเป็น...,ชาวนามีทุ่งรูปสี่เหลี่ยมผืนผ้าที่ยาวเป็นสองเท่...,ให้ความกว้างของสนามเป็น \( w \) เมตร ดังนั้นคว...,id_9998,synthetic_amc,None,None,"[{'role': 'system', 'content': 'You are given ..."


In [9]:

from datasets import Dataset

chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

prepared_dataset["text"] = tokenizer.apply_chat_template(prepared_dataset["Messages"].values.tolist(), tokenize = False)
prepared_dataset = Dataset.from_pandas(prepared_dataset)
prepared_dataset = prepared_dataset.shuffle(seed = 3407)
prepared_dataset

Dataset({
    features: ['reannotated_assistant_content', 'problem', 'solution', 'id', 'source', 'verified', 'quality_metrics', 'Messages', 'text'],
    num_rows: 10000
})

Let's see the first transformed row:

In [10]:
prepared_dataset[0]

{'reannotated_assistant_content': '<think>ก่อนอื่น ฉันต้องกำหนดความชันของเส้นที่กำหนด \\( x - 2y + 1 = 0 \\) ในการทำเช่นนี้ ฉันจะเขียนสมการใหม่ในรูปแบบความชัน-จุดตัดแกน \\( y = mx + b \\) โดยเริ่มจากสมการ: \\[ x - 2y + 1 = 0 \\] ฉันจะแก้หา \\( y \\): \\[ -2y = -x - 1 \\\\ y = \\frac{1}{2}x + \\frac{1}{2} \\] จากนี้ ฉันเห็นว่าความชัน \\( m \\) ของเส้นที่กำหนดคือ \\( \\frac{1}{2} \\) เนื่องจากเส้นที่ต้องการตั้งฉากกับเส้นนี้ ความชัน \\( m&#39; \\) จะต้องเป็นไปตาม: \\[ m \\times m&#39; = -1 \\\\ \\frac{1}{2} \\times m&#39; = -1 \\\\ m&#39; = -2 \\] ตอนนี้ ฉันมีความชันของเส้นที่ต้องการและจุด \\( (2, -1) \\) ที่เส้นผ่าน ฉันจะใช้รูปแบบจุด-ความชันของเส้นตรงเพื่อหาสมการ: \\[ y - y_1 = m&#39;(x - x_1) \\\\ y - (-1) = -2(x - 2) \\\\ y + 1 = -2x + 4 \\\\ y = -2x + 3 \\] สุดท้าย ฉันจะจัดเรียงสมการนี้ให้เป็นรูปแบบมาตรฐาน: \\[ 2x + y = 3 \\] ดังนั้นสมการของเส้นตรงที่ต้องการคือ \\( 2x + y - 3 = 0 \\)</think> ไทย ในการหาสมการของเส้นตรง \\( l \\) ที่ผ่านจุด \\( (2, -1) \\) และตั้งฉากกับเส้นตรง \\( x - 2

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [11]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = prepared_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        max_steps = 15,
        # num_train_epochs = 1, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        dataset_num_proc = 1,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.634200
10,0.519600
15,0.464800
20,0.471800
25,0.561200
30,0.473400
35,0.476000
40,0.453800
45,0.471100
50,0.495200


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for reasoning inference are `temperature = 0.6, top_p = 0.95, top_k = 20`

For normal chat based inference, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [15]:
FastLanguageModel.for_inference(model)

messages = [
    {"role" : "user", "content" : "จงหาจำนวนเต็ม x และ y ทั้งหมดที่ x^2 + y^2 = 5"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Enable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

โอเค ฉันมีปัญหาตรงนี้: ฉันต้องหาจำนวนเต็ม x และ y ทั้งหมดที่ x² + y² = 5 โอเค ฉันจะลองคิดดูว่าจะทำอย่างไร ฉันรู้ว่าจำนวนเต็มคือจำนวนที่ไม่ใช่เศษส่วน ซึ่งรวมถึงจำนวนเต็มบวก จำนวนเต็มลบ และศูนย์ ดังนั้น x และ y จึงเป็นจำนวนเต็ม ดังนั้น ฉันจึงต้องค้นหาคู่ (x, y) ที่เมื่อคูณกันแล้วจะได้ 5 ดังนั้น ฉันจึงต้องค้นหาจำนวนเต็มสองจำนวนที่เมื่อคูณกันแล้วจะได้ 5 แต่เดี๋ยวก่อน 5 คือจำนวนเฉพาะ ซึ่งหมายความว่ามันสามารถแยกตัวประกอบได้เฉพาะ 1 และ 5 ดังนั้น จำนวนเต็มที่เป็นไปได้ที่จะคูณกันแล้วจะได้ 5 คือ 1, -1, 5 และ -5 แต่เนื่องจากกำลังสองของจำนวนเต็มใดๆ ก็ตามจะเป็นจำนวนเต็มบวก ดังนั้น จำนวนเต็มลบจึงไม่สามารถเป็นคำตอบได้ เนื่องจากกำลังสองของจำนวนเต็มลบจะเป็นจำนวนเต็มบวก ดังนั้น ฉันจึงสามารถลดจำนวนตัวเลือกเหลือ 1 และ 5 ได้ ดังนั้น จำนวนเต็มที่เป็นไปได้ที่จะคูณกันแล้วจะได้ 5 คือ 1 และ 5 แต่เนื่องจากกำลังสองของ 1 คือ 1 และกำลังสองของ 5 คือ 25 ซึ่งไม่ใช่ 5 ดังนั้น จำนวนเต็มที่เป็นไปได้ที่จะคูณกันแล้วจะได้ 5 คือ 1 และ 5 แต่กำลังสองของ 1 คือ 1 และกำลังสองของ 5 คือ 25 ซึ่งไม่ใช่ 5 ดังนั้น จำนวนเต็มที่เป็นไปได้

### Exercise 1: Ask the model a challenging math problem. You may increase the value of `max_new_tokens` if the model's answer is too short. Does the model solve your problem correctly?

In [ ]:
### YOUR CODE HERE ###



**Answer to Exercise 1**

-

### Visualizing Attention

### Exercise 2: Fill in the ##TODO## part in the `visualize_attention()` function below to calculate the product between the query matrix `Q` and the key matrix `K`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define the helper function to perform the rotation
def rotate_half(x):
    """Rotates half the hidden dimensions of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# Define a global dictionary to store the attention matrix
attention_viz_data = {}

def visualize_attention(model, tokenizer, prompt, layer_number):
    """
    Visualizes the QK^T attention matrix for a given prompt and layer,

    Args:
        model: The Unsloth Qwen3 14B model.
        tokenizer: The tokenizer for the model.
        prompt (str): The input text prompt.
        layer_number (int): The decoder layer number to visualize.
    """
    attention_viz_data.clear()

    # 1. Get a reference to the specific attention layer
    try:
        attention_layer = model.base_model.model.model.layers[layer_number].self_attn
    except AttributeError as e:
        print(f"Could not find the specified layer. Please double-check the model architecture path. Error: {e}")
        return

    # 2. Store the original forward method
    original_forward = attention_layer.forward

    def new_forward(self, hidden_states, *args, **kwargs):
        """A new forward method to capture the QK^T matrix."""

        bsz, q_len, _ = hidden_states.size()

        # Project to Q, K
        Q_all_heads = self.q_proj(hidden_states)
        K_all_heads = self.k_proj(hidden_states)

        # Reshape for multi-head attention. Note the different head counts.
        Q_all_heads = Q_all_heads.view(bsz, q_len, self.config.num_attention_heads, self.head_dim).transpose(1, 2)
        K_all_heads = K_all_heads.view(bsz, q_len, self.config.num_key_value_heads, self.head_dim).transpose(1, 2)

        # Get rotary embeddings
        cos, sin = self.rotary_emb(K_all_heads, seq_len=q_len)

        # Apply rotary embeddings directly
        Q_all_heads = (Q_all_heads * cos) + (rotate_half(Q_all_heads) * sin)
        K_all_heads = (K_all_heads * cos) + (rotate_half(K_all_heads) * sin)

        # Repeat the key heads to match the number of query heads.
        num_key_value_groups = self.config.num_attention_heads // self.config.num_key_value_heads
        if num_key_value_groups > 1:
            K_all_heads = K_all_heads.repeat_interleave(num_key_value_groups, dim=1)

        # Now, the dimensions match for matrix multiplication:
        # Q_all_heads: [1, 40, 29, 128] = [batch_size, num_heads, seq_len, head_dim]
        # K_all_heads: [1, 40, 29, 128] = [batch_size, num_heads, seq_len, head_dim]

        Q = Q_all_heads[0, 0]  # Queries of the first head
        K = K_all_heads[0, 0]  # Keys of the first head

        ########## TODO: CALCULATE QK^T ############
        attention =

        # Store the matrix for visualization
        attention_viz_data['qk_t'] = attention.detach()

        # Call the original forward method to ensure the model's computation continues normally
        return original_forward(hidden_states, *args, **kwargs)

    # 3. Monkey-patch the layer's forward method
    try:
        attention_layer.forward = new_forward.__get__(attention_layer, type(attention_layer))

        # 4. Prepare and run the model
        inputs = tokenizer(prompt, return_tensors="pt")
        input_ids = inputs.input_ids.to(model.device)

        with torch.no_grad():
            _ = model(input_ids)

    finally:
        # 5. Restore the original forward method
        attention_layer.forward = original_forward

    # 6. Visualize the captured data
    if 'qk_t' in attention_viz_data:
        # We visualize the scores for the first head of the first batch item
        qk_matrix = attention_viz_data['qk_t'].cpu().float().numpy()

        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
        tokens = [token[1:] if token[0] == 'Ġ' else token for token in tokens]

        plt.figure(figsize=(12, 10))
        im = plt.imshow(qk_matrix, cmap='viridis', interpolation='nearest')
        plt.colorbar(im, label='Attention Score (Logits)')
        plt.xticks(ticks=np.arange(len(tokens)), labels=tokens, rotation=90)
        plt.yticks(ticks=np.arange(len(tokens)), labels=tokens)
        plt.xlabel("Key Tokens")
        plt.ylabel("Query Tokens")
        plt.title(f"QK^T Matrix Visualization for Layer {layer_number}, Head 0")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Failed to capture the QK^T matrix for layer {layer_number}.")

In [ ]:
example_text = "Find all integers x and y such that x^2 - y^2 = 5."

visualize_attention(model, tokenizer, example_text, 0)

### Exercise 3: Choose your own example text. Find **two** pairs of two distinct words with the highest attention values. Why do you think those two pairs have the highest attention values?

In [ ]:
### YOUR CODE HERE ###




**Answers to Exercise 3**

-

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False:
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False: # Pushing to HF Hub
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")
